In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

  Using cached uvicorn-0.49.0-py3-none-any.whl.metadata (6.7 kB)
   ---------------------------------------- 0.0/557.4 kB ? eta -:--:--
   ------------------ --------------------- 262.1/557.4 kB ? eta -:--:--
   ---------------------------------------- 557.4/557.4 kB 1.7 MB/s  0:00:00
   ---------------------------------------- 0.0/603.9 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/603.9 kB ? eta -:--:--
   ---------------------------------------- 603.9/603.9 kB 2.0 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 2.1 MB/s eta 0:00:01
   ------------- -------------------------- 0.8/2.4 MB 1.8 MB/s eta 0:00:01
   ----------------- ---------------------- 1.0/2.4 MB 1.5 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 1.6 MB/s eta 0:00:01
   ------------------------------- -------- 1.8/2.4

  You can safely remove it manually.


## Injestion pipeline:
### Coverting Data into Documents 
#### Documents is datatype of Langchain

In [36]:
from langchain_core.documents import Document

In [37]:
sample_doc = Document(
    page_content="Hello World",
    metadata={"Source": "https://www.google.com"}
)

In [38]:
sample_doc

Document(metadata={'Source': 'https://www.google.com'}, page_content='Hello World')

In [39]:
# # text data -> Document
# from langchain_community.document_loaders.text import TextLoader

# loader = TextLoader("Data/sample1.txt", encoding="utf-8")

# document = loader.load()
# document

In [40]:
# # pdf data -> Document

# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("Data/Research.pdf")

# document = pdf_loader.load()

# document 

In [41]:
 # Data => Documents  usning function
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [42]:
def load_all_pdfs():
    folder_path = "Data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [43]:
all_pdf_documents = load_all_pdfs()

Ignoring wrong pointing object 8 0 (offset 0)


total pdfs: 2
total pages: 24


### Converting the Documents into Chunks:

In [44]:
# !pip install langchain_text_splitters

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [46]:
chunks = split_docs(all_pdf_documents)

In [47]:
len(chunks)

268

### Embeddings of chunks:

In [48]:
from sentence_transformers import SentenceTransformer

In [49]:
# creating class to reuse it later in the code instead of writing the whole code again and again
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings 

In [50]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


C:\Users\DELL\AppData\Local\Temp\ipykernel_5288\2383332230.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


## Storing to vector DB
#### We will use the vector Store which s=is small scale version of vector DB

In [51]:
import chromadb
import uuid

In [52]:
class VectorStoreManager:
    def __init__(self, persist_directory="Data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None  #an entity that helps in connecting with vecoer store

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [53]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 268


In [54]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

embeddings shape: (268, 384)
total documents added in vector store= 268
docs in collection: 536


# Retrieval Pipeline

In [55]:
from sklearn.metrics.pairwise import cosine_similarity

In [56]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [57]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [60]:
rag_retriever.retrieve("Language Models")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_56041afa-3805-42ce-9f76-e42cc883398e',
  'document': '[5] S. Borgeaud, A. Mensch, J. Hoffmann, T. Cai, E. Rutherford, K. Milli-\ncan, G. B. Van Den Driessche, J.-B. Lespiau, B. Damoc, A. Clarket al.,\n“Improving language models by retrieving from trillions of tokens,”\nin International conference on machine learning . PMLR, 2022, pp.\n2206–2240.\n[6] L. Ouyang, J. Wu, X. Jiang, D. Almeida, C. Wainwright, P. Mishkin,\nC. Zhang, S. Agarwal, K. Slama, A. Ray et al. , “Training language\nmodels to follow instructions with human feedback,” Advances in',
  'metadata': {'moddate': '2024-03-28T00:54:45+00:00',
   'page_label': '17',
   'source': 'Data\\Research.pdf',
   'subject': '',
   'author': '',
   'keywords': '',
   'doc_index': 167,
   'title': '',
   'producer': 'pdfTeX-1.40.25',
   'trapped': '/False',
   'total_pages': 21,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'content_length': 491,
   'creatio

# Integrate with LLMs